# Wykład 3 — Redukcja wymiarowości i klasteryzacja
### Laboratorium wizualne · Google Colab

**Cel:** zobaczyć, *co robią* algorytmy i jak korzystać z gotowych implementacji `scikit-learn` oraz `umap-learn`. **Nie implementujemy algorytmów od zera.**

Uruchamiaj komórki po kolei. W komórkach z suwakami zmieniaj parametry i obserwuj wykresy. Wykresy używają prawdziwych etykiet tylko do oceny i porównania — algorytmy klasteryzacji ich nie otrzymują.


In [ ]:
%pip -q install umap-learn ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris, load_wine, load_digits, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import linkage, dendrogram
from ipywidgets import interact, interactive, IntSlider, FloatSlider, Dropdown
import umap
sns.set_theme(style='whitegrid', context='notebook')
SEED=42
rng=np.random.default_rng(SEED)
def scatter(ax,X,labels=None,title='',centers=None):
    ax.scatter(X[:,0],X[:,1],c=labels,cmap='tab10' if labels is not None else None,s=22,alpha=.8,edgecolors='none')
    if centers is not None: ax.scatter(centers[:,0],centers[:,1],marker='X',s=240,c='black',edgecolors='white',linewidths=1.5)
    ax.set_title(title);ax.set_xlabel('Wymiar 1');ax.set_ylabel('Wymiar 2')
print('Gotowe!')

## 1. Czy wszystkie cechy mają podobną skalę?
K-Means, PCA i DBSCAN opierają się na odległościach / wariancji. Zobacz, jak jedna cecha o dużej skali może zdominować wynik.

In [ ]:
X,y=make_blobs(n_samples=450,centers=3,cluster_std=1.0,random_state=SEED)
X_stretched=X*np.array([30,1])
fig,axs=plt.subplots(1,3,figsize=(16,4))
scatter(axs[0],X,y,'Oryginalne dane')
scatter(axs[1],X_stretched,y,'Jedna cecha × 30')
scatter(axs[2],StandardScaler().fit_transform(X_stretched),y,'Po StandardScaler')
plt.tight_layout();plt.show()

## 2. PCA — kierunek największej wariancji
Suwak zmienia kąt prostej, na którą rzutujemy punkty. Zwróć uwagę, przy którym kącie wariancja projekcji jest największa. Czerwona linia pokazuje kierunek wybrany przez `PCA` z biblioteki.

In [ ]:
X0=rng.normal(size=(280,2)) @ np.array([[3.1,1.2],[0.2,0.55]])
X0-=X0.mean(axis=0)
pca=PCA(n_components=2).fit(X0)
@interact(kat=IntSlider(value=30,min=0,max=180,step=5,description='Kąt (°)'))
def pca_angle(kat):
    v=np.array([np.cos(np.deg2rad(kat)),np.sin(np.deg2rad(kat))]);z=X0@v
    fig,axs=plt.subplots(1,2,figsize=(12,4))
    axs[0].scatter(*X0.T,alpha=.45,s=15)
    axs[0].plot([-5*v[0],5*v[0]],[-5*v[1],5*v[1]],color='orange',lw=3,label='Wybrany kierunek')
    q=pca.components_[0];axs[0].plot([-5*q[0],5*q[0]],[-5*q[1],5*q[1]],'r--',lw=2,label='PCA 1')
    axs[0].axis('equal');axs[0].legend();axs[0].set_title('Rzutowanie danych')
    axs[1].hist(z,bins=25,color='steelblue');axs[1].set_title(f'Wariancja rzutu: {np.var(z):.2f} (maksimum: {pca.explained_variance_[0]:.2f})')
    plt.tight_layout();plt.show()

## 3. PCA — ile informacji zachowujemy?
Na rzeczywistym zbiorze Wine sprawdzamy `explained_variance_ratio_`.

In [ ]:
wine=load_wine(); Xw=StandardScaler().fit_transform(wine.data)
p=PCA().fit(Xw)
fig,axs=plt.subplots(1,2,figsize=(13,4))
axs[0].bar(np.arange(1,len(p.explained_variance_ratio_)+1),p.explained_variance_ratio_)
axs[0].set(xlabel='Numer komponentu',ylabel='Udział wyjaśnionej wariancji',title='Scree plot')
axs[1].plot(np.arange(1,len(p.explained_variance_ratio_)+1),np.cumsum(p.explained_variance_ratio_),marker='o')
axs[1].axhline(.9,color='red',ls='--',label='90%');axs[1].legend()
axs[1].set(xlabel='Liczba komponentów',ylabel='Skumulowana wariancja',title='Ile komponentów wystarczy?')
plt.tight_layout();plt.show()

## 4. PCA na odręcznych cyfrach — rekonstrukcja obrazów
Zmieniaj liczbę komponentów. Obraz po prawej jest rekonstrukcją po kompresji PCA, a nie wynikiem modelu generatywnego.

In [ ]:
digits=load_digits();Xd=digits.data.astype(float)
@interact(komponenty=IntSlider(value=10,min=1,max=64,step=1,description='PCA'))
def reconstruct_digits(komponenty):
    model=PCA(n_components=komponenty).fit(Xd)
    rec=model.inverse_transform(model.transform(Xd[:8]))
    fig,axs=plt.subplots(2,8,figsize=(14,4))
    for i in range(8):
        axs[0,i].imshow(Xd[i].reshape(8,8),cmap='gray',vmin=0,vmax=16)
        axs[1,i].imshow(rec[i].reshape(8,8),cmap='gray',vmin=0,vmax=16)
        for r in range(2):axs[r,i].axis('off')
    axs[0,0].set_title('Oryginały',loc='left');axs[1,0].set_title('PCA',loc='left')
    fig.suptitle(f'{komponenty} komponentów · wyjaśniona wariancja {model.explained_variance_ratio_.sum():.1%}')
    plt.tight_layout();plt.show()

## 5. PCA — przestrzeń cech rzeczywistych danych
Etykiety kolorują wykres *po redukcji*; PCA nie korzysta z etykiet.

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(17,4))
for ax,(name,data) in zip(axs,[('Iris',load_iris()),('Wine',load_wine()),('Breast Cancer',load_breast_cancer())]):
    z=PCA(n_components=2).fit_transform(StandardScaler().fit_transform(data.data))
    scatter(ax,z,data.target,name)
plt.tight_layout();plt.show()

## 6. PCA kontra t-SNE kontra UMAP — rzeczywiste cyfry
Porównaj globalną geometrię i lokalne sąsiedztwa. t-SNE i UMAP służą często do wizualizacji, a nie do mierzenia odległości pomiędzy odległymi grupami. Obliczenia mogą potrwać chwilę.

In [ ]:
X=StandardScaler().fit_transform(Xd)
embeddings={
    'PCA':PCA(n_components=2).fit_transform(X),
    't-SNE':TSNE(n_components=2,perplexity=30,init='pca',learning_rate='auto',random_state=SEED).fit_transform(X),
    'UMAP':umap.UMAP(n_components=2,n_neighbors=15,min_dist=.1,random_state=SEED).fit_transform(X)
}
fig,axs=plt.subplots(1,3,figsize=(17,5))
for ax,(name,z) in zip(axs,embeddings.items()):scatter(ax,z,digits.target,name)
plt.tight_layout();plt.show()

## 7. t-SNE — jak perplexity zmienia obraz?
Na tych samych danych możesz otrzymać bardzo różne wizualizacje. **Nie interpretuj wielkości i odległości między grupami jako wiernych odległości w oryginalnej przestrzeni.**

In [ ]:
@interact(perplexity=Dropdown(options=[5,15,30,50,80],value=30,description='Perplexity'))
def tsne_param(perplexity):
    z=TSNE(n_components=2,perplexity=perplexity,init='pca',learning_rate='auto',random_state=SEED).fit_transform(X)
    fig,ax=plt.subplots(figsize=(7,5));scatter(ax,z,digits.target,f't-SNE · perplexity={perplexity}');plt.show()

## 8. UMAP — lokalność i upakowanie
`n_neighbors` wpływa na równowagę lokalnej i szerszej struktury, a `min_dist` na stopień upakowania punktów. Przeliczenie może potrwać chwilę.

In [ ]:
@interact(n_neighbors=Dropdown(options=[5,15,40,80],value=15,description='Sąsiedzi'),min_dist=Dropdown(options=[0.0,0.1,0.5,0.9],value=0.1,description='min_dist'))
def umap_param(n_neighbors,min_dist):
    z=umap.UMAP(n_neighbors=n_neighbors,min_dist=min_dist,random_state=SEED).fit_transform(X)
    fig,ax=plt.subplots(figsize=(7,5));scatter(ax,z,digits.target,f'UMAP · neighbors={n_neighbors}, min_dist={min_dist}');plt.show()

## 9. K-Means — gotowy algorytm z biblioteki
Suwaki pokazują, jak wybór liczby klastrów i rozrzutu danych zmienia wynik. Krzyżyki oznaczają centroidy.

In [ ]:
@interact(k=IntSlider(min=2,max=8,value=3,description='K'),rozrzut=FloatSlider(min=.3,max=3,value=1,step=.2,description='Rozrzut'))
def kmeans_live(k,rozrzut):
    x,y=make_blobs(n_samples=450,centers=3,cluster_std=rozrzut,random_state=SEED)
    model=KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(x)
    fig,axs=plt.subplots(1,2,figsize=(12,4))
    scatter(axs[0],x,y,'Rzeczywiste grupy (tylko porównanie)')
    scatter(axs[1],x,model.labels_,f'K-Means, K={k}',model.cluster_centers_)
    plt.tight_layout();plt.show()

## 10. Jak wybrać K? Metoda łokcia i silhouette score
Sprawdź, czy obie metryki sugerują podobną liczbę grup. W praktyce żadna nie gwarantuje „prawdziwego” K.

In [ ]:
xb,yb=make_blobs(n_samples=600,centers=4,cluster_std=.85,random_state=SEED)
ks=range(2,11); inertias=[];sil=[]
for k in ks:
    m=KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(xb)
    inertias.append(m.inertia_);sil.append(silhouette_score(xb,m.labels_))
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(list(ks),inertias,'o-');axs[0].set(title='Elbow / inertia',xlabel='K',ylabel='Suma kwadratów odległości')
axs[1].plot(list(ks),sil,'o-');axs[1].set(title='Silhouette',xlabel='K',ylabel='Silhouette score')
plt.tight_layout();plt.show()

## 11. K-Means vs DBSCAN vs Agglomerative — różne geometrie
Trzy algorytmy na tych samych danych: kuliste skupiska, księżyce, pierścienie i dane z szumem. **To najważniejsza demonstracja na zajęciach.**

In [ ]:
datasets={
    'Blobs':make_blobs(n_samples=450,centers=3,cluster_std=.65,random_state=SEED),
    'Moons':make_moons(n_samples=450,noise=.07,random_state=SEED),
    'Circles':make_circles(n_samples=450,noise=.055,factor=.45,random_state=SEED),
}
fig,axs=plt.subplots(3,4,figsize=(17,12))
for i,(name,(x,y)) in enumerate(datasets.items()):
    x=StandardScaler().fit_transform(x)
    labels=[y,KMeans(n_clusters=len(np.unique(y)),n_init=10,random_state=SEED).fit_predict(x),DBSCAN(eps=.25,min_samples=5).fit_predict(x),AgglomerativeClustering(n_clusters=len(np.unique(y))).fit_predict(x)]
    for j,(lab,title) in enumerate(zip(labels,['Etykiety referencyjne','K-Means','DBSCAN','Agglomerative'])):
        scatter(axs[i,j],x,lab,f'{name} · {title}'+(f' · ARI {adjusted_rand_score(y,lab):.2f}' if j else ''))
plt.tight_layout();plt.show()

## 12. DBSCAN — parametry i wykrywanie szumu
Czarne punkty na osobnym wykresie to obserwacje oznaczone `-1` (szum). Zmieniaj `eps` i `min_samples`.

In [ ]:
@interact(eps=FloatSlider(min=.05,max=.8,step=.05,value=.25,description='eps'),min_samples=IntSlider(min=2,max=20,value=5,description='min_samples'),zbior=Dropdown(options=['Moons','Circles','Blobs'],value='Moons'))
def dbscan_live(eps,min_samples,zbior):
    x,y=datasets[zbior];x=StandardScaler().fit_transform(x)
    labels=DBSCAN(eps=eps,min_samples=min_samples).fit_predict(x)
    fig,axs=plt.subplots(1,2,figsize=(12,4))
    scatter(axs[0],x,labels,f'DBSCAN · klastry: {len(set(labels)-{-1})}')
    axs[1].scatter(x[labels!=-1,0],x[labels!=-1,1],s=18,alpha=.5,label='Przypisane')
    axs[1].scatter(x[labels==-1,0],x[labels==-1,1],s=30,c='black',label='Szum (-1)')
    axs[1].set_title(f'Punkty szumu: {(labels==-1).sum()}');axs[1].legend()
    plt.tight_layout();plt.show()

## 13. Klasteryzacja hierarchiczna — dendrogram
Dendrogram przedstawia, w jakiej kolejności łączone są grupy. Zmień metodę `linkage`.

In [ ]:
xh,yh=make_blobs(n_samples=65,centers=4,cluster_std=.8,random_state=SEED)
@interact(metoda=Dropdown(options=['single','complete','average','ward'],value='ward',description='Linkage'))
def hierarchy(metoda):
    Z=linkage(xh,method=metoda)
    fig,axs=plt.subplots(1,2,figsize=(14,4))
    scatter(axs[0],xh,yh,'Dane wejściowe')
    dendrogram(Z,ax=axs[1],no_labels=True,color_threshold=None)
    axs[1].set(title=f'Dendrogram: {metoda}',xlabel='Obserwacje',ylabel='Odległość połączenia')
    plt.tight_layout();plt.show()

## 14. Realne dane Wine — porównanie klasteryzacji
Standaryzujemy dane, klasteryzujemy w **oryginalnej przestrzeni cech po skalowaniu**, a PCA używamy **tylko do narysowania wyniku**. Nie myl klasteryzacji w 2D z klasteryzacją w pełnej przestrzeni.

In [ ]:
data=load_wine();x=StandardScaler().fit_transform(data.data);z=PCA(n_components=2).fit_transform(x)
models={'K-Means':KMeans(n_clusters=3,n_init=10,random_state=SEED),
        'DBSCAN':DBSCAN(eps=3.0,min_samples=5),
        'Agglomerative':AgglomerativeClustering(n_clusters=3)}
fig,axs=plt.subplots(1,4,figsize=(18,4))
scatter(axs[0],z,data.target,'Znane klasy (referencja)')
for ax,(name,m) in zip(axs[1:],models.items()):
    lab=m.fit_predict(x)
    scatter(ax,z,lab,f'{name} · ARI={adjusted_rand_score(data.target,lab):.2f}')
plt.tight_layout();plt.show()

## 15. Realne dane Digits — klastry vs cyfry
Czy algorytm bez etykiet potrafi odkryć wszystkie cyfry? Kolory klastra są **arbitralne** i nie muszą odpowiadać numerom cyfr.

In [ ]:
z=embeddings['PCA'];labels=KMeans(n_clusters=10,n_init=10,random_state=SEED).fit_predict(X)
fig,axs=plt.subplots(1,2,figsize=(13,5))
scatter(axs[0],z,digits.target,'Prawdziwe cyfry · PCA 2D')
scatter(axs[1],z,labels,f'K-Means w 64D · ARI={adjusted_rand_score(digits.target,labels):.2f}')
plt.tight_layout();plt.show()
ct=pd.crosstab(pd.Series(digits.target,name='Cyfra'),pd.Series(labels,name='Klaster'))
plt.figure(figsize=(10,5));sns.heatmap(ct,annot=True,fmt='d',cmap='Blues');plt.title('Cyfra vs klaster — to nie jest macierz pomyłek klasyfikatora');plt.show()

## 16. Pułapka: wymuszone K klastry na losowym szumie
K-Means *zawsze* przypisze punkty do K grup, nawet jeśli w danych nie ma naturalnych klastrów.

In [ ]:
noise=rng.uniform(-3,3,size=(500,2))
fig,axs=plt.subplots(1,3,figsize=(15,4))
scatter(axs[0],noise,title='Jednorodny szum (bez naturalnych grup)')
for ax,k in zip(axs[1:],[3,6]):
    m=KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(noise)
    scatter(ax,noise,m.labels_,f'K-Means K={k}',m.cluster_centers_)
plt.tight_layout();plt.show()

## 17. Mini-zadania do wykonania na lekcji

1. W PCA zwiększ liczbę komponentów z 2 do 40. Kiedy cyfry stają się czytelne?
2. W t-SNE zmień `perplexity`. Czy położenie grup pozostaje takie samo?
3. Na zbiorze **Moons** spróbuj uzyskać dwa klastry metodą DBSCAN, a następnie K-Means.
4. Na **Circles** znajdź `eps`, które połączy oba pierścienie w jeden klaster. Dlaczego tak się dzieje?
5. Porównaj dendrogram dla `single` i `ward`.
6. Na **Wine** sprawdź, czy klastry odpowiadają prawdziwym klasom.
7. Wyjaśnij, dlaczego K-Means znalazł klastry w losowym szumie.

### Najważniejsze wnioski
- PCA zachowuje możliwie dużą wariancję w liniowym rzucie, niekoniecznie informację o klasach.
- t-SNE i UMAP są użyteczne do wizualizacji, ale nie gwarantują zachowania globalnych odległości.
- K-Means preferuje zwarte skupiska i wymaga podania K.
- DBSCAN rozpoznaje skupiska gęstościowe i szum, ale jest wrażliwy na skalę oraz parametry.
- Wynik klasteryzacji zależy od cech, metryki odległości, skalowania i przyjętych założeń.
